In [ ]:
import os
import re
import json
import torch
import pickle
import numpy as np
import os.path as osp
import soundfile as sf
from copy import copy
from peft import PeftModel
from transformers import (
    GenerationConfig, 
    AutoTokenizer, 
    AutoConfig,
    set_seed
)
from safetensors.torch import load_file
from vllm import LLM, SamplingParams
from models_llama import LlamaForCausalLM_val as ModelForCausalLM
from tqdm import tqdm
from pdb import set_trace

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = "cuda:0"

exp_id = "v13_02"
exp_step = "160000"
exp_dir = f"/proj/speech/kqian/text2hpc/Exps/{exp_id}"
lora_weights = os.path.join(exp_dir, f"checkpoint-{exp_step}")
tokenizer_path = os.path.join(exp_dir, "tokenizer")
eot_id = '<|eot_id|>'
base_model = "/proj/speech/kqian/pretrained_models/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
model_config = AutoConfig.from_pretrained(exp_dir)

prs_dim = 5
n_prs_bins = 513
len_tokenizer = len(tokenizer)

try:
    model = LLM(
        model=os.path.join(lora_weights, 'merged_model'), skip_tokenizer_init=True, #dtype='float32'
    )
    with open(osp.join(lora_weights, "merged_model", "config.json"), "r") as ff:
        model_config = json.load(ff)
except ValueError:
    print('Merge model ......')
    model = ModelForCausalLM.from_pretrained(base_model,
                                             config=model_config,
                                             device_map="auto", 
                                             torch_dtype=torch.float32)
    model.resize_token_embeddings(len_tokenizer)
    model = PeftModel.from_pretrained(
                model,
                lora_weights,
                torch_dtype=torch.float32,
            )
    model = model.merge_and_unload()

    state_dict = load_file(osp.join(lora_weights, 'adapter_model.safetensors'))
    param1 = state_dict['base_model.model.model.embed_tokens.original_embed.weight']
    param2 = state_dict['base_model.model.model.embed_tokens.new_embed.weight']
    embed_tokens_weight = torch.cat((param1, param2), dim=0)
    param1 = state_dict['base_model.model.lm_head.original_head.weight']
    param2 = state_dict['base_model.model.lm_head.new_head.weight']
    lm_head_weight = torch.cat((param1, param2), dim=0)
    with torch.no_grad():
        model.model.embed_tokens.weight.copy_(embed_tokens_weight)
        model.lm_head.weight.copy_(lm_head_weight)
    
    model.save_pretrained(osp.join(lora_weights, 'merged_model'))

    with open(osp.join(lora_weights, "merged_model", "config.json"), "r") as ff:
        model_config = json.load(ff) 
    model_config['architectures'] = ['LlamaForCausalLM']
    with open(osp.join(lora_weights, "merged_model", "config.json"), "w") as ff:
        json.dump(model_config, ff, indent=2)
        
    del model
    torch.cuda.empty_cache()
    model = LLM(
        model=os.path.join(lora_weights, 'merged_model'), skip_tokenizer_init=True, #dtype='float32'
    )

In [ ]:
import os
import re
import json
import torch
import pickle
import numpy as np
import os.path as osp
from copy import copy
from peft import PeftModel
from transformers import (
    GenerationConfig, 
    AutoTokenizer, 
    AutoConfig,
    set_seed
)

base_model = "/proj/speech/kqian/pretrained_models/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.padding_side = "left"  # Allow batched inference
tokenizer.init_kwargs["padding_side"] = tokenizer.padding_side
tokenizer.add_tokens(['[NUM]','[SPK_EMB]','<SIL>','[SEP_1]','[SEP_2]'], special_tokens=True)
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

print(len(tokenizer))

num_tks = [f"<|num_tk_{i}|>" for i in range(513)]
tokenizer.add_special_tokens(
    {"additional_special_tokens": num_tks}, 
    replace_additional_special_tokens=False
)


num_tks = [f"<|spk_tk_{i}|>" for i in range(512)]
tokenizer.add_special_tokens(
    {"additional_special_tokens": num_tks}, 
    replace_additional_special_tokens=False
)

tokenizer.save_pretrained(os.path.join('hf_ckpt/llm/', 'tokenizer'))

In [ ]:
from transformers import AutoTokenizer, set_seed
from vllm import LLM, SamplingParams
from llm.models_llama import LlamaForCausalLM_val as ModelForCausalLM
from llm.utils import (
    TemperatureLogitsProcessor,
    RangeConstrainedLogitsProcessor,
    TemplateLogitsProcessor,
    parse_prs_token_sequence,
    replace_num_tokens
)
from utils import build_pattern

device = "cuda:0"

model_path = "/proj/long-multi/kqian/text2hpc/Exps/v10_13/checkpoint-160000/merged_model"
tokenizer_path = "hf_ckpt/llm/tokenizer"

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

model = LLM(
    model=model_path,
    tokenizer=tokenizer_path,
)

In [ ]:
import yaml
import torch
import numpy as np
from collections import OrderedDict

from data.utils import _text_to_phonemes, find_consecutive_indices

from tts.utils import (
    TextCleaner,
    _punctuation, _pad,
    dicts as sym2id,
    recursive_munch, 
    length_to_mask
)

from tts.PLBERT.util import load_plbert
from tts.models import build_model, load_ASR_models, load_F0_models

from IPython.lib.display import Audio
import IPython.display as ipd

config_path = 'tts/Configs/config_ft.yml'
config = yaml.safe_load(open(config_path))

text_cleaner = TextCleaner()

id2sym = {v:k for k,v in sym2id.items()}

# load pretrained ASR model
ASR_config = config.get('ASR_config', False)
text_aligner = load_ASR_models(ASR_config)

# load pretrained F0 model
pitch_extractor = load_F0_models()

# load BERT model
BERT_path = config.get('PLBERT_dir', False)
plbert = load_plbert(BERT_path)

model_params = recursive_munch(config['model_params'])
tts_model = build_model(model_params, text_aligner, pitch_extractor, plbert)
_ = [tts_model[key].eval() for key in tts_model]
_ = [tts_model[key].to(device) for key in tts_model]

params_whole = torch.load("/proj/long-multi/kqian/StyleTTS2/Models/libriheavy_v01_v2/epoch_2nd_00028.pth", 
                          map_location='cpu', weights_only=True)
params = params_whole['net']

for key in tts_model:
    if key in params:
        print('%s loaded' % key)
        try:
            tts_model[key].load_state_dict(params[key])
        except:
            state_dict = params[key]
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:] # remove `module.`
                new_state_dict[name] = v
            # load params
            tts_model[key].load_state_dict(new_state_dict, strict=False)
_ = [tts_model[key].eval() for key in tts_model]

In [ ]:
from evaluation_temp import data_b as data
from collections import defaultdict

all_inputs = []
for idx, item in enumerate(data, 1):
    original = item["original"]
    changes = item["changes"]
    components = item["components"]
    for change_type, (changed_sentence, _) in changes.items():
        conversation = [
            f'"{original}." Tom said.',
            f'"Did you say {changed_sentence}?" Jerry asked, apparently not paying attention.',
            f'"No, I said {original}!"',
        ]
        typ = change_type[:3]
        if typ in ['Sub', 'Ver']:
            all_inputs.append(conversation)

In [ ]:
from collections import deque
from copy import copy
from tts.utils import inference
import pickle
import re

from tqdm import tqdm
tqdm._instances.clear()


with open('../process_data/spk_embs/libriheavy_v01_v2/12139.pkl','rb') as ff:
    spk_emb = pickle.load(ff)
spk_emb = spk_emb.to(device)
sty_emb = torch.zeros_like(spk_emb)

silence = np.zeros(5000, dtype=np.float32)

f0_med = f'<|spk_tk_{220}|>'

temperature_logit_processor = TemperatureLogitsProcessor(0.1, 0.7, 128262, 128262+513)

seed = 0
set_seed(seed)


for inputs in all_inputs:

    combined = []
    model_content = deque()
    model_content_size = 0
    instruction = f"Spin a narrative {f0_med}: "
    
    for i, input_sent in enumerate(inputs):
        model_content.append(f'{input_sent}[SEP_1]')
        content = instruction + ''.join(model_content)
        model_message = {'role': 'assistant', 'content': content}
        prompt = tokenizer.apply_chat_template([model_message], tokenize=False, add_generation_prompt=False)
        prompt = re.sub(r'<\|eot_id\|>(?!.*<\|eot_id\|>)', '', prompt, flags=re.DOTALL)

        template, pattern = build_pattern(tokenizer, input_sent)
        pattern_constrained_processor = TemplateLogitsProcessor(
            template, pattern, 
            128262, 128262+513,
        )
        #print(template)

        sampling_params = SamplingParams(
            repetition_penalty=1.0,
            top_p=0.95,
            top_k=250,
            stop='[SEP_2]',
            include_stop_str_in_output=True,
            seed=seed,
            max_tokens=1000,
            detokenize=True,
            skip_special_tokens=False,
            logits_processors=[
                temperature_logit_processor,
                #pattern_constrained_processor
            ]
        )
        
        generation_output = model.generate(
            prompts=[prompt],
            sampling_params=sampling_params,
        )
    
        output_txt = generation_output[0].outputs[0].text
        assert template == replace_num_tokens(output_txt)
        output_nums = parse_prs_token_sequence(output_txt)
        model_content.append(output_txt)
        model_content_size += 1
        assert len(model_content) == 2*model_content_size
        while model_content_size >= 6:
            model_content.popleft()
            model_content.popleft()
            model_content_size -= 1
    
        extracted_numbers = []
        for prs_vec in output_nums:
            if len(prs_vec) == 1:
                new_prs = [512 for _ in range (5)]
                new_prs[0] = prs_vec[0]
            else:
                new_prs = copy(prs_vec)
            extracted_numbers.append(new_prs)
    
        # text to speech
        ps = _text_to_phonemes(input_sent.strip())
        tokens = text_cleaner(ps, 'llm')
        tokens.insert(0, 0)
        prsinf = np.array(extracted_numbers)
        
        is_sil = [1 if id2sym[tk] in _punctuation+_pad else 0 for tk in tokens]
        indices_word = find_consecutive_indices(is_sil)
        
        tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
        boundaries = np.array(indices_word)[:, 1:].T
        
        y_pred = inference(tts_model, tokens, spk_emb, sty_emb, prsinf, boundaries, device)
        
        combined.append(y_pred)
        combined.append(silence)
                
    combined = np.concatenate(combined[:-1])
    ipd.display(Audio(combined, rate=24000))

    #break